In [64]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, roc_auc_score
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [40]:
def calc_metrics(y_true, pred):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=np.array([0,1])).ravel()
    
    return pd.DataFrame({'acc': accuracy_score(y_true, pred),
                         'f1': f1_score(y_true, pred),
                         'tn': tn,
                         'fp': fp,
                         'fn': fn,
                         'tp': tp}, index=np.array([0]))


In [23]:
old_data_dir = '/media/ssd1/huong/PCR-huong/data'
new_data_dir = os.path.join(old_data_dir, 'new_data')
old_modeloutput_dir = os.path.join(old_data_dir, 'model_outputs')
new_modeloutput_dir = os.path.join(new_data_dir, 'model_outputs')

In [24]:
curve_df = pd.read_hdf(os.path.join(old_data_dir,'data.h5'), key='curve_data')
sample_info = pd.read_hdf(os.path.join(old_data_dir,'data.h5'), key='sample_info')
igi_gene_call = pd.read_hdf(os.path.join(old_data_dir,'data.h5'), key='igi_gene_call')

join_df = (curve_df
           .merge(sample_info, how='inner', on=['well_position','pcr_plate'])
           .merge(igi_gene_call, how ='inner', on=['pcr_plate','sample_id','target']))
thermo_sample_ids = join_df[join_df.target == 'MS2'].sample_id.unique()
join_df['test_kit'] = 'LuNER'
join_df.loc[join_df.sample_id.isin(thermo_sample_ids),'test_kit'] = 'Thermo'

In [25]:
clinical_pred = pd.read_csv(os.path.join(old_modeloutput_dir, 'fusion_vit_delta64_clinical_pred_df.csv'))

In [26]:
clinical_pred_df = (clinical_pred
                    .merge(join_df[(join_df.cycle_no == 1)], 
                           on='curve_idx'))
clinical_pred_df['encoded_igi_call'] = 1*(clinical_pred_df['igi_call'] == 'Positive')

In [51]:
gene_index_full = (clinical_pred_df.loc[~clinical_pred_df.target.isin(['RnaseP','MS2']), 
                                        ['curve_idx','target','sample_id','outputs','encoded_igi_call']]
                 .merge(clinical_pred_df.loc[~clinical_pred_df.target.isin(['RnaseP','MS2']),
                                             ['curve_idx','target','sample_id','encoded_igi_call']],
                        on='sample_id', suffixes=['_index','_other']))
gene_index_full = gene_index_full[gene_index_full.target_index != gene_index_full.target_other]
print(gene_index_full.shape)
gene_index_full.head()

(58212, 8)


,curve_idx_index,target_index,sample_id,outputs,encoded_igi_call_index,curve_idx_other,target_other,encoded_igi_call_other
1,115142,N gene,S1077292,0.000406,0,114758,E gene,0
2,114758,E gene,S1077292,0.000222,0,115142,N gene,0
5,17445,N gene,S455929,0.356368,1,17061,E gene,0
6,17061,E gene,S455929,0.000186,0,17445,N gene,1
9,103552,N gene,S1088502,0.000128,0,103168,E gene,0


In [52]:
gene_index_df = (gene_index_full
                 .groupby(['sample_id','curve_idx_index','target_index','outputs','encoded_igi_call_index'])
                 .agg(other_target_avg = ('encoded_igi_call_other','mean'),
                      other_target_major = ('encoded_igi_call_other',lambda x: 1*(np.mean(x) >= 0.5)))
                 .reset_index())

### Regression ver 1

In [68]:
lm_model = ols(formula='other_target_major ~ outputs', data=gene_index_df)
results = lm_model.fit()
print(results.summary())
roc_auc_score(gene_index_df.other_target_major, results.predict())

                            OLS Regression Results                            
Dep. Variable:     other_target_major   R-squared:                       0.771
Model:                            OLS   Adj. R-squared:                  0.771
Method:                 Least Squares   F-statistic:                 1.480e+05
Date:                Thu, 11 Apr 2024   Prob (F-statistic):               0.00
Time:                        12:52:57   Log-Likelihood:                 24242.
No. Observations:               44031   AIC:                        -4.848e+04
Df Residuals:                   44029   BIC:                        -4.846e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0138      0.001     19.767      0.0

0.9469518277041957

In [69]:
lm_model = ols(formula='other_target_major ~ encoded_igi_call_index', data=gene_index_df)
results = lm_model.fit()
print(results.summary())
roc_auc_score(gene_index_df.other_target_major, results.predict())

                            OLS Regression Results                            
Dep. Variable:     other_target_major   R-squared:                       0.772
Model:                            OLS   Adj. R-squared:                  0.772
Method:                 Least Squares   F-statistic:                 1.494e+05
Date:                Thu, 11 Apr 2024   Prob (F-statistic):               0.00
Time:                        12:53:18   Log-Likelihood:                 24398.
No. Observations:               44031   AIC:                        -4.879e+04
Df Residuals:                   44029   BIC:                        -4.877e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  0

0.935276682020931

#### Regression ver 2

In [73]:
gene_index_df_ver2 = (gene_index_df
                      .merge(clinical_pred_df.loc[clinical_pred_df.target.isin(['RnaseP','MS2']),
                                                  ['sample_id','outputs','encoded_igi_call']],
                             on='sample_id', suffixes=['_index','_control']))
gene_index_df_ver2

,sample_id,curve_idx_index,target_index,outputs_index,encoded_igi_call_index,other_target_avg,other_target_major,outputs_control,encoded_igi_call
0,S079403,128445,N gene,0.000261,0,0.0,0,1.000000,1
1,S079403,128829,ORF1ab,0.000714,0,0.0,0,1.000000,1
2,S079403,129213,S gene,0.009346,0,0.0,0,1.000000,1
3,S079404,128541,N gene,0.000179,0,0.0,0,0.999999,1
4,S079404,128925,ORF1ab,0.001655,0,0.0,0,0.999999,1
...,...,...,...,...,...,...,...,...,...
44026,S995298,63137,N gene,1.000000,1,1.0,1,1.000000,1
44027,S995299,62801,E gene,0.001353,0,0.0,0,1.000000,1
44028,S995299,63185,N gene,0.000066,0,0.0,0,1.000000,1
44029,S995300,62849,E gene,0.001618,0,0.0,0,1.000000,1


In [72]:
lm_model = ols(formula='other_target_major ~ outputs_index + outputs_control + outputs_index*outputs_control', data=gene_index_df_ver2)
results = lm_model.fit()
print(results.summary())
roc_auc_score(gene_index_df.other_target_major, results.predict())

                            OLS Regression Results                            
Dep. Variable:     other_target_major   R-squared:                       0.771
Model:                            OLS   Adj. R-squared:                  0.771
Method:                 Least Squares   F-statistic:                 4.942e+04
Date:                Thu, 11 Apr 2024   Prob (F-statistic):               0.00
Time:                        12:54:34   Log-Likelihood:                 24274.
No. Observations:               44031   AIC:                        -4.854e+04
Df Residuals:                   44027   BIC:                        -4.850e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

0.9495104039948424

In [74]:
lm_model = ols(formula='other_target_major ~ encoded_igi_call_index + encoded_igi_call + encoded_igi_call_index*encoded_igi_call', data=gene_index_df_ver2)
results = lm_model.fit()
print(results.summary())
roc_auc_score(gene_index_df.other_target_major, results.predict())

                            OLS Regression Results                            
Dep. Variable:     other_target_major   R-squared:                       0.773
Model:                            OLS   Adj. R-squared:                  0.773
Method:                 Least Squares   F-statistic:                 4.991e+04
Date:                Thu, 11 Apr 2024   Prob (F-statistic):               0.00
Time:                        12:55:41   Log-Likelihood:                 24440.
No. Observations:               44031   AIC:                        -4.887e+04
Df Residuals:                   44027   BIC:                        -4.884e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

0.9394658173981584

In [55]:
clinical_pred_df = (clinical_pred
                    .merge(join_df[(join_df.cycle_no == 1) &
                                   (~join_df.target.isin(['MS2','RnaseP']))], 
                           on='curve_idx'))
clinical_pred_df['encoded_igi_call'] = 1*(clinical_pred_df['igi_call'] == 'Positive')
clinical_pred_df['major_igi_call'] = clinical_pred_df.groupby('sample_id').encoded_igi_call.transform(lambda x: np.round(np.mean(x)))
igi_pos_frac = sum(clinical_pred_df.encoded_igi_call == 1)/clinical_pred_df.shape[0]
print('IGI external positive prediction:', igi_pos_frac)

IGI external positive prediction: 0.09173082600894825


In [56]:
model_dict = {'thres':[], 'acc':[],'fnr':[], 'fpr':[],'pos_frac':[]}
for i in np.arange(0,1,0.001):
    clinical_pred_df['pred'] = 1*(clinical_pred_df.outputs >= i)
    tn, fp, fn, tp = confusion_matrix(clinical_pred_df.groundtruth_label, clinical_pred_df.pred, labels=np.array([0,1])).ravel()
    
    model_dict['thres'].append(i)
    model_dict['acc'].append((tp+tn)/(tp+fp+tn+fn))
    model_dict['fnr'].append(fn/(fn+tp))
    model_dict['fpr'].append(fp/(fp+tn))
    model_dict['pos_frac'].append((tp+fp)/(tp+fp+tn+fn))
    
model_dict_df = pd.DataFrame(model_dict)
    

In [51]:
model_dict_df[(model_dict_df.pos_frac <= igi_pos_frac + 0.00003) & (model_dict_df.pos_frac >= igi_pos_frac - 0.00003)]

,thres,acc,fnr,fpr,pos_frac
485,0.485,0.708541,0.063148,0.311802,0.362938
486,0.486,0.708541,0.063148,0.311802,0.362938
487,0.487,0.708541,0.063148,0.311802,0.362938
488,0.488,0.708541,0.063148,0.311802,0.362938
489,0.489,0.708541,0.063148,0.311802,0.362938
490,0.490,0.708525,0.063340,0.311802,0.362923


In [57]:
clinical_pred_df['pred'] = 1*(clinical_pred_df.outputs >= 0.098)
# clinical_pred_df['pred'] = 1*(clinical_pred_df.outputs >= 0.485)

In [58]:
calc_metrics(clinical_pred_df.major_igi_call, clinical_pred_df.pred)

,acc,f1,tn,fp,fn,tp
0,0.982898,0.902119,39808,569,184,3470


In [59]:
calc_metrics(clinical_pred_df.major_igi_call, clinical_pred_df.encoded_igi_call)

,acc,f1,tn,fp,fn,tp
0,0.990302,0.944495,39971,406,21,3633
